# Data Transformation - Kairos Project

**Team:** Ilariê

**Objective:** This notebook details the data transformation phase of the Kairos Project, following the data cleaning already performed. The goal is to prepare the datasets for modeling by applying categorical variable encoding techniques.

**Datasets**:
1. SERVICE_ORDER_BASE.xlsx (Service order details)
2. VEHICLES_BASE.xlsx (Vehicle master data)

## 0. Configuration and Data Loading

This initial section establishes the environment, imports the necessary libraries, and loads the datasets that will be used.

### Importing libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Pandas display settings for better visualization
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded successfully!")

### Loading datasets

In [ ]:
try:
    df_service = pd.read_excel('data/SERVICE_ORDER_BASE.xlsx')
    df_vehicles = pd.read_excel('data/VEHICLES_BASE.xlsx')

    datasets = {
        'Service Orders': df_service,
        'Vehicle Master Data': df_vehicles
    }
    print("Datasets loaded successfully!")

except Exception as e:
    print(f"Error on loading datasets: {e}")
    print("Please, ensure the data files are on the correct directory.")

## 1. Data Integration for a Single Dataset (Merge)
**Objective:** To build a predictive model that understands maintenance patterns ("what," "when," "how much it cost") in relation to vehicle characteristics ("which model," "which year," "which manufacturer"), it is essential to consolidate the information into a single dataset.

In [ ]:
print(f"Records in df_service before merge: {len(df_service)}")
print(f"Records in df_vehicles: {len(df_vehicles)}")

df_merged = pd.merge(df_service, df_vehicles, on='ASSET CODE', how='left', suffixes=('', '_vehicle'))

print(f"Records in df_merged after merge: {len(df_merged)}")

print("\nOverview of merged dataframe (df_merged):")
display(df_merged.head())

## 2. Feature Engineering
With the unified dataset, we can create new variables (features) that capture richer and more useful information for the model.

### 2.1. Extracting Temporal Patterns
We extract information from date columns to capture seasonality and age.

In [ ]:
# Ensuring date columns are in datetime format
df_merged['SERVICE ORDER ORIGINAL DATE'] = pd.to_datetime(df_merged['SERVICE ORDER ORIGINAL DATE'])
df_merged['ASSET PURCHASE DATE'] = pd.to_datetime(df_merged['ASSET PURCHASE DATE'])

# 1. Calculating vehicle age at time of service
df_merged['VEHICLE_AGE_AT_SERVICE'] = (df_merged['SERVICE ORDER ORIGINAL DATE'] - df_merged['ASSET PURCHASE DATE']).dt.days / 365.25

# 2. Extracting components from the work order date
df_merged['SERVICE_YEAR'] = df_merged['SERVICE ORDER ORIGINAL DATE'].dt.year
df_merged['SERVICE_MONTH'] = df_merged['SERVICE ORDER ORIGINAL DATE'].dt.month
df_merged['SERVICE_DAY_OF_WEEK'] = df_merged['SERVICE ORDER ORIGINAL DATE'].dt.dayofweek # Monday=0, Sunday=6

print("New temporal features created:")
display(df_merged[['SERVICE ORDER ORIGINAL DATE', 'ASSET PURCHASE DATE', 'VEHICLE_AGE_AT_SERVICE', 'SERVICE_YEAR', 'SERVICE_MONTH']].head())

### 2.2. Handling the Original Date Column
**Objective:** After extracting the relevant information (age, year, month, etc.), the original date column (SERVICE ORDER ORIGINAL DATE) may become redundant. Many models cannot interpret datetime columns directly, and keeping them can add noise. Therefore, it is a good practice to remove them after feature engineering.

In [ ]:
# Removing the original date columns to avoid redundancy
df_merged.drop(['SERVICE ORDER ORIGINAL DATE', 'ASSET PURCHASE DATE'], axis=1, inplace=True)
print("Original date columns removed.")

### 2.3. Creating Rates and Relationships
We create features that represent important business relationships, such as cost per unit or per kilometer.

In [ ]:
df_merged['COST_PER_UNIT'] = df_merged['GRAND TOTAL'] / df_merged['PRODUCT QUANTITY'].replace(0, np.nan)
df_merged['COST_PER_KM'] = df_merged['GRAND TOTAL'] / df_merged['COUNTER  OF SERVICE ORDER'].replace(0, np.nan)

df_merged['COST_PER_UNIT'].fillna(0, inplace=True)
df_merged['COST_PER_KM'].fillna(0, inplace=True)

print("New rate features created:")
display(df_merged[['GRAND TOTAL', 'PRODUCT QUANTITY', 'COUNTER  OF SERVICE ORDER', 'COST_PER_UNIT', 'COST_PER_KM']].head())

## 3. Encoding Categorical Variables

**Technical Objective**: Convert categorical variables into a numerical format so that machine learning algorithms can process them.

### 3.1. One-Hot Encoding

For nominal variables with no intrinsic order. This technique creates new binary columns for each category, avoiding the interpretation of a non-existent order.

Variables to consider: **'MODEL TYPE DESCRIPTION', 'FAMILY NAME', 'MANUFACTURER NAME', 'PREVENTIVE_CORRECTIVE MAINTENANCE'**.

In [ ]:
cols_to_onehot = [
    'MODEL TYPE DESCRIPTION',
    'FAMILY NAME',
    'MANUFACTURER NAME',
    'PREVENTIVE_CORRECTIVE MAINTENANCE'
]

# Removing columns that do not exist in the dataframe
cols_to_onehot = [col for col in cols_to_onehot if col in df_merged.columns]

print(f"Columns to One-Hot Encoding: {cols_to_onehot}")

# Applying One-Hot Encoding
df_merged_encoded = pd.get_dummies(df_merged, columns=cols_to_onehot, prefix=cols_to_onehot, drop_first=True)

print("\nDimensions before encoding:", df_merged.shape)
print("Dimensions after encoding:", df_merged_encoded.shape)

### 3.2. Binary Encoding
For variables with only two categories. Uses 0 and 1, which is computationally efficient.

Variables to consider: **TIRE ('YES'/'NO'), ASSET STATUS ('ACTIVE'/'INACTIVE'), MAINTENANCE TYPE ('INTERNAL'/'EXTERNAL'), PREVENTIVE_CORRECTIVE MAINTENANCE ('PREVENTIVE'/'CORRECTIVE')**.

In [ ]:
if 'ASSET STATUS' in df_merged_encoded.columns:
    df_merged_encoded['ASSET_STATUS_ENCODED'] = df_merged_encoded['ASSET STATUS'].map({'ACTIVE': 1, 'INACTIVE': 0})
    df_merged_encoded.drop('ASSET STATUS', axis=1, inplace=True)
    print("\n'ASSET STATUS' codificado (Binário).")

if 'MAINTENANCE TYPE' in df_merged_encoded.columns:
    df_merged_encoded['MAINTENANCE_TYPE_ENCODED'] = df_merged_encoded['MAINTENANCE TYPE'].map({'INTERNAL': 1, 'EXTERNAL': 0})
    df_merged_encoded.drop('MAINTENANCE TYPE', axis=1, inplace=True)
    print("'MAINTENANCE TYPE' codificado (Binário).")

print("Binarized columns:")
display(df_merged_encoded[['ASSET_STATUS_ENCODED', 'MAINTENANCE_TYPE_ENCODED']].head())

### 3.3. Ordinal Encoding
For categorical variables with an intrinsic order. Assigns integers that respect the order of the categories.

Variable to consider: **TIER ('TIER 1', 'TIER 2', etc.)**.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

if 'TIER' in df_merged_encoded.columns:
    tier_order = ['TIER 1 - T1', 'TIER 2 - T2'] # explicit order based on glossary passed by Fadel
    ordinal_encoder = OrdinalEncoder(categories=[tier_order])
    df_merged_encoded['TIER_ENCODED'] = ordinal_encoder.fit_transform(df_merged_encoded[['TIER']])
    df_merged_encoded.drop('TIER', axis=1, inplace=True)
    print("'TIER' codificado (Ordinal).")

## 4. Normalization of Numerical Variables

Technical Objective: Adjust the scale of numerical variables to prevent one from dominating calculations in magnitude-sensitive algorithms.

Variables to consider: **GRAND TOTAL, PRODUCT QUANTITY, UNIT VALUE, SERVICE ORDER COUNTER, MANUFACTURE YEAR**.

### 4.1. Z-score Normalization (Standardization)
Justification for Choice:

Z-score Normalization (StandardScaler): Transforms the data to have a mean of 0 and a standard deviation of 1. It is a robust choice, especially if outlier treatment (capping) has already been performed. It is less sensitive to remaining outliers than Min-Max and is a requirement for many algorithms (e.g., SVMs, Logistic Regression with regularization).

Min-Max Normalization (MinMaxScaler): Rescales the data to a fixed interval [0, 1]. It is useful for algorithms that do not assume a specific distribution, such as Neural Networks.

Decision: Z-score (StandardScaler)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Selecting only numeric columns for normalization (excluding IDs and already encoded columns)
numeric_cols = df_merged_encoded.select_dtypes(include=np.number).columns.tolist()
# Remove columns that are not continuous features, such as codes or binary flags
cols_to_exclude = [col for col in df_merged_encoded.columns if '_ENCODED' in col] # Excludes already encoded columns
cols_to_exclude.extend([col for col in numeric_cols if 'CODE' in col.upper()]) # Excludes numeric codes

numeric_features = [col for col in numeric_cols if col not in cols_to_exclude]

print("Numeric columns to be normalized (Z-score):")
print(numeric_features)

# applying Z-score normalization
scaler = StandardScaler()
df_merged_encoded[numeric_features] = scaler.fit_transform(df_merged_encoded[numeric_features])

print("\nVisualization of normalized data:")
display(df_merged_encoded[numeric_features].describe())